# SDH exp_016 — OOF class-mass calibration
exp14 모델은 고정하고 OOF 기반 클래스 확률 보정만 검증한다. 셀을 위에서 아래로 실행한다.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'experiments').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'experiments').exists():
    raise RuntimeError('저장소 내부에서 실행해 주세요.')
EXP13_DIR = PROJECT_ROOT / 'experiments/SDH/exp_013_standalone_pipeline_audit'
EXP14_DIR = PROJECT_ROOT / 'experiments/SDH/exp_014_focal_lgbm_specialists'
EXP16_DIR = PROJECT_ROOT / 'experiments/SDH/exp_016_oof_class_calibration'
RESULTS_DIR = EXP16_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for path in (EXP13_DIR, EXP14_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import standalone_pipeline as p13
import lgbm_experiment as exp14
print('project root:', PROJECT_ROOT)

In [ ]:
data_dir = PROJECT_ROOT / 'data/raw'
train = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')
sample_submission = pd.read_csv(data_dir / 'sample_submission.csv')
genes = [column for column in train if column not in ('ID', 'SUBCLASS')]
labels = train['SUBCLASS'].to_numpy()
classes = np.asarray(sorted(np.unique(labels)))
SEEDS = (42, 52, 62)
LR_WEIGHT, LGBM_WEIGHT = 0.80, 0.20
MAIN_CASE = exp14.main_cases()['main_01_multiclass_balanced']
SPECIALIST_CASE = exp14.specialist_cases()['spec_07_both_hard_predicted']
print(train.shape, test.shape, len(classes))

## 1. exp14 OOF 3-seed 재현
각 outer fold의 train에서만 피처와 유사 class pair를 정한다. 시간이 가장 오래 걸리는 셀이다.

In [ ]:
oof_by_seed = {}
lr_oof_by_seed = {}
lgbm_oof_by_seed = {}
oof_rows = []
for seed in SEEDS:
    print(f'\n===== OOF seed {seed} =====')
    prepared, seed_labels, seed_classes = exp14.prepare_seed(train, genes, seed=seed)
    assert np.array_equal(seed_labels, labels)
    assert np.array_equal(seed_classes, classes)
    lr_result = exp14.evaluate_lr_reference(prepared, labels, classes, seed=seed)
    main_result = exp14.evaluate_main_case(prepared, labels, classes, MAIN_CASE, seed=seed)
    specialist_folds = exp14.fit_specialist_probabilities(prepared, labels, seed=seed)
    specialist_result = exp14.apply_specialist_case(
        main_result, specialist_folds, labels, SPECIALIST_CASE,
    )
    probability = LR_WEIGHT * lr_result.probability + LGBM_WEIGHT * specialist_result.probability
    np.testing.assert_allclose(probability.sum(axis=1), 1.0, atol=1e-6)
    prediction = classes[np.argmax(probability, axis=1)]
    score = f1_score(labels, prediction, average='macro')
    lr_oof_by_seed[seed] = lr_result.probability
    lgbm_oof_by_seed[seed] = specialist_result.probability
    oof_by_seed[seed] = probability
    oof_rows.append({'seed': seed, 'baseline_f1': score})
    print('exp14 blend:', score)
oof_baseline = pd.DataFrame(oof_rows)
display(oof_baseline)
print('mean:', oof_baseline['baseline_f1'].mean())

## 2. 저차원 class-mass 보정
26개 weight를 직접 맞추지 않고 alpha 하나만 탐색한다. clip 범위도 0.8~1.25로 제한한다.

In [ ]:
ALPHA_GRID = np.round(np.arange(0.0, 1.01, 0.1), 2)
WEIGHT_CLIP = (0.80, 1.25)

def fit_mass_weights(probabilities, y, class_names, alpha):
    true_mass = np.array([(y == name).mean() for name in class_names], dtype=np.float64)
    predicted_mass = np.asarray(probabilities, dtype=np.float64).mean(axis=0)
    ratio = np.divide(true_mass, predicted_mass, out=np.ones_like(true_mass), where=predicted_mass > 0)
    return np.clip(ratio ** alpha, *WEIGHT_CLIP)

def apply_mass_weights(probabilities, weights):
    calibrated = np.asarray(probabilities, dtype=np.float64) * weights
    calibrated /= calibrated.sum(axis=1, keepdims=True)
    return calibrated

def macro_f1(probabilities):
    return f1_score(labels, classes[np.argmax(probabilities, axis=1)], average='macro')

In [ ]:
holdout_rows = []
for holdout_seed in SEEDS:
    fit_seeds = [seed for seed in SEEDS if seed != holdout_seed]
    fit_probability = np.concatenate([oof_by_seed[seed] for seed in fit_seeds], axis=0)
    fit_labels = np.tile(labels, len(fit_seeds))
    alpha_scores = []
    for alpha in ALPHA_GRID:
        weights = fit_mass_weights(fit_probability, fit_labels, classes, alpha)
        scores = [macro_f1(apply_mass_weights(oof_by_seed[seed], weights)) for seed in fit_seeds]
        alpha_scores.append((float(np.mean(scores)), -float(alpha), float(alpha), weights))
    _, _, selected_alpha, selected_weights = max(alpha_scores, key=lambda row: (row[0], row[1]))
    baseline = macro_f1(oof_by_seed[holdout_seed])
    calibrated = macro_f1(apply_mass_weights(oof_by_seed[holdout_seed], selected_weights))
    holdout_rows.append({
        'holdout_seed': holdout_seed, 'fit_seeds': str(fit_seeds),
        'selected_alpha': selected_alpha, 'baseline_f1': baseline,
        'calibrated_f1': calibrated, 'delta': calibrated - baseline,
    })
holdout_result = pd.DataFrame(holdout_rows)
holdout_result.to_csv(RESULTS_DIR / 'seed_holdout_calibration.csv', index=False)
display(holdout_result)
CALIBRATION_PASS = bool((holdout_result['delta'] > 0).all())
print('mean delta:', holdout_result['delta'].mean())
print('minimum delta:', holdout_result['delta'].min())
print('verdict:', 'PASS' if CALIBRATION_PASS else 'FAIL — 제출하지 않음')

## 3. 최종 alpha 확정
세 holdout에서 모두 개선된 경우에만 진행한다. 선택 alpha의 median을 사용하고 전체 OOF로 class weight를 한 번 계산한다.

In [ ]:
if CALIBRATION_PASS:
    FINAL_ALPHA = float(holdout_result['selected_alpha'].median())
    all_oof_probability = np.concatenate([oof_by_seed[seed] for seed in SEEDS], axis=0)
    all_oof_labels = np.tile(labels, len(SEEDS))
    FINAL_CLASS_WEIGHTS = fit_mass_weights(all_oof_probability, all_oof_labels, classes, FINAL_ALPHA)
else:
    FINAL_ALPHA = 0.0
    FINAL_CLASS_WEIGHTS = np.ones(len(classes), dtype=np.float64)
    print('class-mass FAIL: 원본 확률을 유지하고 다음 실험으로 진행합니다.')
final_weight_table = pd.DataFrame({'class': classes, 'weight': FINAL_CLASS_WEIGHTS})
display(final_weight_table.sort_values('weight', ascending=False))
print('FINAL_ALPHA:', FINAL_ALPHA)

## 4. LR/LGBM 혼합비 seed-holdout 재검증
exp14의 LGBM 20%를 기준으로 5%~35%를 탐색한다. 두 seed에서 weight 하나를 선택하고 남은 seed에서 기존 20%보다 좋아지는지 확인한다.

In [ ]:
if 'lr_oof_by_seed' not in globals() or not lr_oof_by_seed:
    raise RuntimeError('1번 exp14 OOF 3-seed 재현 셀부터 다시 실행해 주세요.')
LGBM_WEIGHT_GRID = np.round(np.arange(0.05, 0.351, 0.025), 3)
BASELINE_LGBM_WEIGHT = 0.20

def blend_probability(seed, lgbm_weight):
    return (1.0 - lgbm_weight) * lr_oof_by_seed[seed] + lgbm_weight * lgbm_oof_by_seed[seed]

blend_holdout_rows = []
for holdout_seed in SEEDS:
    fit_seeds = [seed for seed in SEEDS if seed != holdout_seed]
    candidates = []
    for weight in LGBM_WEIGHT_GRID:
        fit_scores = [macro_f1(blend_probability(seed, weight)) for seed in fit_seeds]
        candidates.append((float(np.mean(fit_scores)), -abs(float(weight) - BASELINE_LGBM_WEIGHT), float(weight)))
    _, _, selected_weight = max(candidates, key=lambda row: (row[0], row[1]))
    baseline = macro_f1(blend_probability(holdout_seed, BASELINE_LGBM_WEIGHT))
    selected = macro_f1(blend_probability(holdout_seed, selected_weight))
    blend_holdout_rows.append({
        'holdout_seed': holdout_seed, 'fit_seeds': str(fit_seeds),
        'selected_lgbm_weight': selected_weight,
        'baseline_20_f1': baseline, 'selected_f1': selected, 'delta': selected - baseline,
    })
blend_holdout = pd.DataFrame(blend_holdout_rows)
blend_holdout.to_csv(RESULTS_DIR / 'seed_holdout_blend_weight.csv', index=False)
display(blend_holdout)
BLEND_WEIGHT_PASS = bool((blend_holdout['delta'] > 0).all())
FINAL_LGBM_WEIGHT = float(blend_holdout['selected_lgbm_weight'].median())
print('mean delta:', blend_holdout['delta'].mean())
print('minimum delta:', blend_holdout['delta'].min())
print('median selected weight:', FINAL_LGBM_WEIGHT)
print('verdict:', 'PASS' if BLEND_WEIGHT_PASS else 'FAIL — 제출하지 않음')

In [ ]:
weight_curve_rows = []
for weight in LGBM_WEIGHT_GRID:
    scores = [macro_f1(blend_probability(seed, weight)) for seed in SEEDS]
    weight_curve_rows.append({
        'lgbm_weight': float(weight), 'mean_f1': float(np.mean(scores)),
        'std_f1': float(np.std(scores)), 'min_f1': float(np.min(scores)),
        **{f'seed_{seed}': score for seed, score in zip(SEEDS, scores)},
    })
weight_curve = pd.DataFrame(weight_curve_rows)
weight_curve.to_csv(RESULTS_DIR / 'blend_weight_curve_3seed.csv', index=False)
display(weight_curve.sort_values(['mean_f1', 'min_f1'], ascending=False))

## 5. 혼합비 제출 생성
혼합비가 세 holdout에서 모두 PASS일 때만 exp14를 전체 train으로 다시 학습한다. 유사 class pair와 모든 피처 통계는 train에서만 정하며 test는 transform/predict에만 사용한다.

In [ ]:
def aligned_probability(model, matrix, class_names):
    raw = np.asarray(model.predict_proba(matrix), dtype=np.float64)
    lookup = {name: index for index, name in enumerate(model.classes_)}
    return raw[:, [lookup[name] for name in class_names]]

def discover_similar_pairs(matrix, names, y, top_n=2):
    gene_columns = np.array([name.startswith('G__') for name in names])
    gene_matrix = matrix[:, gene_columns]
    centroids = []
    for class_name in classes:
        centroid = np.asarray(gene_matrix[y == class_name].mean(axis=0)).ravel()
        norm = np.linalg.norm(centroid)
        centroids.append(centroid / norm if norm > 0 else centroid)
    candidates = []
    for left_index, left in enumerate(classes):
        for right_index in range(left_index + 1, len(classes)):
            candidates.append((-float(centroids[left_index] @ centroids[right_index]), left, classes[right_index]))
    candidates.sort()
    return tuple((left, right) for _, left, right in candidates[:top_n])

def fit_binary_specialist(x_train, y, x_test, pair, seed):
    mask = np.isin(y, pair)
    model = LGBMClassifier(
        objective='binary', boosting_type='gbdt', reg_alpha=0.0, reg_lambda=0.0,
        importance_type='gain', class_weight='balanced', random_state=seed, n_jobs=-1,
        deterministic=True, force_col_wise=True, verbosity=-1,
        **exp14._specialist_parameters(seed),
    )
    model.fit(x_train[mask], y[mask])
    return aligned_probability(model, x_test, np.asarray(pair))

def hard_route(main_probability, pairs, specialist_probabilities):
    probability = main_probability.copy()
    original_prediction = classes[np.argmax(main_probability, axis=1)]
    lookup = {name: index for index, name in enumerate(classes)}
    for pair, specialist in zip(pairs, specialist_probabilities):
        columns = [lookup[pair[0]], lookup[pair[1]]]
        mass = probability[:, columns].sum(axis=1)
        mask = np.isin(original_prediction, pair)
        probability[mask, columns[0]] = mass[mask] * specialist[mask, 0]
        probability[mask, columns[1]] = mass[mask] * specialist[mask, 1]
    np.testing.assert_allclose(probability.sum(axis=1), 1.0, atol=1e-6)
    return probability

In [ ]:
if not BLEND_WEIGHT_PASS:
    raise RuntimeError('혼합비 seed-holdout 3개가 모두 개선되지 않아 제출을 중단합니다.')
seed_test_probabilities = []
for seed in SEEDS:
    print(f'\n===== full-train seed {seed} =====')
    x_train, x_test, feature_names, audit = p13.build_design_matrices(
        train[genes], test[genes], labels, genes, seed=seed, use_fixed_contrast=False,
    )
    assert audit['raw_train_test_concat'] is False
    assert not any(name.startswith(('C__', 'D__exact_')) for name in feature_names)
    lr_model = p13.make_model(seed)
    lr_model.fit(x_train, labels)
    lr_probability = aligned_probability(lr_model, x_test, classes)
    main_model = LGBMClassifier(**exp14._main_parameters(seed, MAIN_CASE, len(classes)))
    main_model.fit(x_train, labels)
    main_probability = exp14._aligned_probability(main_model, x_test, classes, focal=False)
    pairs = discover_similar_pairs(x_train, feature_names, labels)
    specialist_probabilities = tuple(
        fit_binary_specialist(x_train, labels, x_test, pair, seed) for pair in pairs
    )
    routed_probability = hard_route(main_probability, pairs, specialist_probabilities)
    seed_test_probabilities.append((1.0 - FINAL_LGBM_WEIGHT) * lr_probability + FINAL_LGBM_WEIGHT * routed_probability)
    print('pairs:', pairs)
final_probability = np.mean(seed_test_probabilities, axis=0)
np.testing.assert_allclose(final_probability.sum(axis=1), 1.0, atol=1e-6)
calibrated_test_probability = apply_mass_weights(final_probability, FINAL_CLASS_WEIGHTS)
calibrated_submission = sample_submission.copy()
calibrated_submission['SUBCLASS'] = classes[np.argmax(calibrated_test_probability, axis=1)]
assert calibrated_submission['ID'].equals(test['ID'])
assert not calibrated_submission['SUBCLASS'].isna().any()
output_path = RESULTS_DIR / f'submission_exp016_lgbm_weight{FINAL_LGBM_WEIGHT:.3f}.csv'
calibrated_submission.to_csv(output_path, index=False)
display(calibrated_submission.head())
print('saved:', output_path)